# Dominick's cereal sample construction

This notebook constructs the cleaned store–UPC–week panel used for demand estimation.

The pipeline:

1. merges movement, product, and store metadata;
2. restricts the data to source-valid observations and stores with price-tier metadata;
3. reconstructs prices for zero-sales observations using same-UPC, same-week, same-tier medians;
4. validates the imputation with held-out observed prices;
5. adds calendar, promotion, and post-promotion variables using past information only;
6. removes one non-cereal UPC and a very small set of cross-sectionally isolated sales anomalies;
7. saves the final demand-estimation sample and audit tables.

`main_data` is the source-valid processed panel. `demand_data` is the eligible demand sample before the sales-anomaly exclusion. `demand_model_data` is the final estimation sample.


## 1. Setup


In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
from dateutil.easter import easter
from IPython.display import display


In [ ]:
# Resolve paths whether the notebook is launched from the repository root
# or from the notebooks directory.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_DIR = PROJECT_ROOT / "data" / "raw"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
TABLE_DIR = PROJECT_ROOT / "results" / "tables"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

MOVEMENT_PATH = RAW_DIR / "wcer.csv"
PRODUCT_PATH = RAW_DIR / "upccer.csv"
STORE_PATH = RAW_DIR / "demo.dta"

for path in [MOVEMENT_PATH, PRODUCT_PATH, STORE_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required source file not found: {path}")


In [ ]:
# Fixed preprocessing choices. Keep exclusions explicit and auditable.
EXCLUDED_WEEKS = {219}
NON_CEREAL_UPCS = {317}

DISCOUNT_THRESHOLD = 0.05
REGULAR_PRICE_WINDOW = "91D"
REGULAR_PRICE_MIN_PERIODS = 4
REGULAR_PRICE_QUANTILE = 0.90

SALES_OUTLIER_MIN_MOVE = 1_000
SALES_OUTLIER_MIN_STORES = 20
SALES_OUTLIER_MIN_P95_RATIO = 10
PARQUET_COMPRESSION = "zstd"
RANDOM_SEED = 42


## 2. Load and merge source data

The movement data are merged with the UPC lookup using a many-to-one restriction. The merge audit verifies product coverage and the uniqueness of store–UPC–week observations.


In [ ]:
movement = pd.read_csv(MOVEMENT_PATH)
products = pd.read_csv(PRODUCT_PATH, encoding="cp1252")
stores = pd.read_stata(STORE_PATH)

# Standardize names before downstream processing.
for frame in [movement, products, stores]:
    frame.columns = frame.columns.str.strip().str.lower()

required_movement = {
    "store", "upc", "week", "move", "qty", "price",
    "sale", "profit", "ok",
}
required_products = {"upc", "descrip", "size"}
required_stores = {"store", "priclow", "pricmed", "prichigh"}

assert required_movement.issubset(movement.columns)
assert required_products.issubset(products.columns)
assert required_stores.issubset(stores.columns)
assert not products["upc"].duplicated().any()


In [ ]:
data = movement.merge(
    products,
    on="upc",
    how="left",
    validate="many_to_one",
    indicator="product_merge",
)

merge_audit = pd.Series(
    {
        "rows": len(data),
        "stores": data["store"].nunique(dropna=True),
        "products": data["upc"].nunique(dropna=True),
        "weeks": data["week"].nunique(dropna=True),
        "store_product_pairs": data[["store", "upc"]].drop_duplicates().shape[0],
        "matched_product_share": data["descrip"].notna().mean(),
        "duplicate_store_upc_week_rows": data.duplicated(
            ["store", "upc", "week"]
        ).sum(),
    },
    name="value",
)

display(merge_audit)

assert data["product_merge"].eq("both").all()
assert merge_audit["duplicate_store_upc_week_rows"] == 0

data = data.drop(columns="product_merge")


## 3. Source-field cleaning and store price tiers

`SALE` is treated as a recorded-promotion flag whenever it is nonmissing. Store price tiers are reconstructed from the mutually exclusive `priclow`, `pricmed`, and `prichigh` indicators. Stores without tier metadata are excluded because tier membership is required for the main price-imputation procedure.


In [ ]:
# Normalize promotion codes and standardize key identifiers.
data["sale"] = (
    data["sale"]
    .astype("string")
    .str.strip()
    .replace("", pd.NA)
)
data["promo_recorded"] = data["sale"].notna()

data["store"] = pd.to_numeric(
    data["store"], errors="raise"
).astype("Int64")
data["week"] = pd.to_numeric(
    data["week"], errors="raise"
).astype("int16")
stores["store"] = pd.to_numeric(
    stores["store"], errors="coerce"
).astype("Int64")

assert data["week"].between(1, 399).all()


In [ ]:
tier_columns = ["priclow", "pricmed", "prichigh"]
stores[tier_columns] = stores[tier_columns].apply(
    pd.to_numeric,
    errors="coerce",
)

stores["tier_indicator_sum"] = stores[tier_columns].fillna(0).sum(axis=1)
if not stores["tier_indicator_sum"].eq(1).all():
    bad_rows = stores.loc[
        ~stores["tier_indicator_sum"].eq(1),
        ["store"] + tier_columns,
    ]
    raise ValueError(f"Invalid price-tier indicators:\n{bad_rows}")

stores["price_tier"] = np.select(
    [
        stores["priclow"].eq(1),
        stores["pricmed"].eq(1),
        stores["prichigh"].eq(1),
    ],
    ["low", "medium", "high"],
    default=pd.NA,
)
stores["price_tier"] = pd.Categorical(
    stores["price_tier"],
    categories=["low", "medium", "high"],
)


In [ ]:
# Duplicate metadata rows are acceptable only when they imply the same tier.
tier_conflicts = stores.groupby(
    "store", observed=True
)["price_tier"].nunique()
assert tier_conflicts.le(1).all()

store_info = (
    stores[["store", "price_tier"]]
    .dropna(subset=["store"])
    .drop_duplicates(subset="store")
)
assert not store_info["store"].duplicated().any()

data = data.merge(
    store_info,
    on="store",
    how="left",
    validate="many_to_one",
    indicator="store_merge",
)

unmatched_stores = sorted(
    data.loc[
        data["price_tier"].isna(), "store"
    ].dropna().unique().tolist()
)


In [ ]:
# Restrict the processed panel before imputation. Invalid observations must
# not serve as price donors or enter past-only price histories.
valid_source_row = (
    data["price_tier"].notna()
    & data["ok"].eq(1)
    & ~data["week"].isin(EXCLUDED_WEEKS)
    & data["qty"].gt(0)
    & data["move"].ge(0)
    & data["store"].notna()
    & data["upc"].notna()
)

source_quality_audit = pd.Series(
    {
        "raw_merged_rows": len(data),
        "movement_stores": data["store"].nunique(),
        "stores_with_price_tier": data.loc[
            data["price_tier"].notna(), "store"
        ].nunique(),
        "rows_with_price_tier": data["price_tier"].notna().sum(),
        "rows_with_ok_1": data["ok"].eq(1).sum(),
        "valid_tier_matched_rows": valid_source_row.sum(),
    },
    name="value",
)

display(source_quality_audit)
print("Stores excluded because price-tier metadata are unavailable:", unmatched_stores)


In [ ]:
main_data = data.loc[valid_source_row].copy()
main_data = main_data.drop(columns="store_merge")

assert main_data["ok"].eq(1).all()
assert not main_data["week"].isin(EXCLUDED_WEEKS).any()
assert main_data["qty"].gt(0).all()
assert main_data["move"].ge(0).all()

# Compact dtypes reduce memory pressure in later rolling operations.
main_data["store"] = main_data["store"].astype("int16")
main_data["move"] = main_data["move"].astype("int32")
main_data["sale"] = main_data["sale"].astype("category")

unused_source_columns = [
    "price_hex", "profit_hex", "com_code", "case", "nitem", "ok",
]
main_data = main_data.drop(
    columns=[c for c in unused_source_columns if c in main_data.columns]
)

del data, movement, products, stores


## 4. Price reconstruction

The recorded unit price is `PRICE / QTY` when the transaction price is positive. For zero-sales observations without a recorded price, the model price is imputed as the contemporaneous median price of the same UPC among source-valid stores in the same price tier. Positive-sales observations with missing prices are not imputed.

The raw `PROFIT` field is retained only where the transaction price is observed. This prevents zero values attached to imputed zero-sales rows from being interpreted as observed margins.


In [ ]:
main_data["unit_price_observed"] = (
    main_data["price"] / main_data["qty"]
).where(
    main_data["price"].gt(0)
    & main_data["qty"].gt(0)
)

price_group = main_data.groupby(
    ["upc", "week", "price_tier"],
    observed=True,
)["unit_price_observed"]

main_data["tier_week_median_price"] = price_group.transform("median")
main_data["tier_week_price_donors"] = (
    price_group.transform("count").astype("int16")
)


In [ ]:
can_impute = (
    main_data["move"].eq(0)
    & main_data["unit_price_observed"].isna()
    & main_data["tier_week_median_price"].notna()
)

main_data["model_unit_price"] = main_data["unit_price_observed"]
main_data.loc[can_impute, "model_unit_price"] = main_data.loc[
    can_impute, "tier_week_median_price"
]

main_data["price_source"] = np.select(
    [main_data["unit_price_observed"].notna(), can_impute],
    ["observed", "tier_median_imputed"],
    default="unresolved",
)
main_data["price_source"] = pd.Categorical(
    main_data["price_source"],
    categories=["observed", "tier_median_imputed", "unresolved"],
)

main_data["price_imputed"] = main_data["price_source"].eq(
    "tier_median_imputed"
)
main_data["gross_margin_pct_observed"] = main_data["profit"].where(
    main_data["unit_price_observed"].notna()
)


In [ ]:
imputed = main_data["price_source"].eq("tier_median_imputed")
observed = main_data["price_source"].eq("observed")
unresolved = main_data["price_source"].eq("unresolved")

price_reconstruction_audit = pd.Series(
    {
        "observed_prices": observed.sum(),
        "imputed_prices": imputed.sum(),
        "unresolved_prices": unresolved.sum(),
        "usable_price_share": main_data["model_unit_price"].notna().mean(),
        "imputed_share_with_at_least_3_donors": main_data.loc[
            imputed, "tier_week_price_donors"
        ].ge(3).mean(),
    },
    name="value",
)

display(price_reconstruction_audit)

assert main_data.loc[imputed, "move"].eq(0).all()
assert main_data.loc[imputed, "model_unit_price"].gt(0).all()
assert np.allclose(
    main_data.loc[observed, "model_unit_price"],
    main_data.loc[observed, "unit_price_observed"],
)
assert main_data.loc[unresolved, "model_unit_price"].isna().all()
assert not (imputed & main_data["move"].gt(0)).any()

# PRICE and QTY are no longer needed after unit-price construction.
main_data = main_data.drop(columns=["price", "qty"])


### 4.1 Held-out validation of tier-median imputation

One observed store price is held out from every UPC–week–tier group containing at least two observed prices. The held-out price is predicted from the median of the remaining stores in that group.


In [ ]:
group_columns = ["upc", "week", "price_tier"]

observed_prices = main_data.loc[
    main_data["unit_price_observed"].notna(),
    group_columns + ["store", "unit_price_observed"],
].copy()

observed_prices["group_size"] = observed_prices.groupby(
    group_columns,
    observed=True,
)["unit_price_observed"].transform("size")

eligible = observed_prices.loc[
    observed_prices["group_size"].ge(2)
].copy()

rng = np.random.default_rng(RANDOM_SEED)
eligible["_random"] = rng.random(len(eligible))

holdout_indices = (
    eligible.sort_values("_random")
    .drop_duplicates(group_columns)
    .index
)


In [ ]:
holdout = observed_prices.loc[holdout_indices].copy()
donor_data = observed_prices.drop(index=holdout_indices)

donor_medians = (
    donor_data.groupby(group_columns, observed=True)
    .agg(
        predicted_price=("unit_price_observed", "median"),
        validation_donors=("unit_price_observed", "size"),
    )
    .reset_index()
)

validation = holdout.merge(
    donor_medians,
    on=group_columns,
    how="left",
    validate="one_to_one",
)

assert validation["predicted_price"].notna().all()
assert validation["validation_donors"].ge(1).all()


In [ ]:
validation["absolute_error"] = (
    validation["predicted_price"]
    - validation["unit_price_observed"]
).abs()
validation["percentage_error"] = (
    validation["predicted_price"]
    - validation["unit_price_observed"]
) / validation["unit_price_observed"]
validation["absolute_percentage_error"] = (
    validation["percentage_error"].abs()
)

validation["donor_group"] = pd.cut(
    validation["validation_donors"],
    bins=[0, 1, 2, float("inf")],
    labels=["1 donor", "2 donors", "3+ donors"],
    include_lowest=True,
)


In [ ]:
validation_summary = pd.Series(
    {
        "validated_prices": len(validation),
        "median_absolute_error": validation["absolute_error"].median(),
        "median_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].median(),
        "mean_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].mean(),
        "mean_percentage_error": validation["percentage_error"].mean(),
        "rmse": np.sqrt(np.mean(validation["absolute_error"] ** 2)),
        "within_5_percent": validation[
            "absolute_percentage_error"
        ].le(0.05).mean(),
        "within_10_percent": validation[
            "absolute_percentage_error"
        ].le(0.10).mean(),
        "p95_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].quantile(0.95),
        "p99_absolute_percentage_error": validation[
            "absolute_percentage_error"
        ].quantile(0.99),
    },
    name="value",
)


In [ ]:
validation_by_donors = (
    validation.groupby("donor_group", observed=True)
    .agg(
        observations=("absolute_percentage_error", "size"),
        median_ape=("absolute_percentage_error", "median"),
        mean_ape=("absolute_percentage_error", "mean"),
        p95_ape=("absolute_percentage_error", lambda x: x.quantile(0.95)),
        within_5_percent=(
            "absolute_percentage_error",
            lambda x: x.le(0.05).mean(),
        ),
        mean_signed_error=("percentage_error", "mean"),
    )
)

validation_by_tier = (
    validation.groupby("price_tier", observed=True)
    .agg(
        observations=("absolute_percentage_error", "size"),
        median_ape=("absolute_percentage_error", "median"),
        mean_ape=("absolute_percentage_error", "mean"),
        p95_ape=("absolute_percentage_error", lambda x: x.quantile(0.95)),
        within_5_percent=(
            "absolute_percentage_error",
            lambda x: x.le(0.05).mean(),
        ),
    )
)

display(validation_summary)
display(validation_by_donors)
display(validation_by_tier)

# Release large validation intermediates before rolling calculations.
del observed_prices, eligible, holdout, donor_data, donor_medians


## 5. Calendar controls

Dominick's weeks run from Thursday through Wednesday. Calendar month is assigned from the week-ending date. Holiday indicators equal one when the holiday date falls within the corresponding Thursday–Wednesday interval.


In [ ]:
def fourth_thursday_of_november(year: int) -> pd.Timestamp:
    # Return the date of U.S. Thanksgiving for a given year.
    november_first = pd.Timestamp(year=year, month=11, day=1)
    days_until_thursday = (3 - november_first.weekday()) % 7
    return november_first + pd.Timedelta(days=days_until_thursday + 21)


week_lookup = pd.DataFrame(
    {"week": np.arange(1, 400, dtype=np.int16)}
)
week_lookup["week_start"] = (
    pd.Timestamp("1989-09-14")
    + pd.to_timedelta((week_lookup["week"] - 1) * 7, unit="D")
)
week_lookup["week_end"] = (
    week_lookup["week_start"] + pd.Timedelta(days=6)
)
week_lookup["calendar_month"] = (
    week_lookup["week_end"].dt.month.astype("int8")
)
week_lookup["time_trend"] = (week_lookup["week"] - 1).astype("int16")


In [ ]:
years = range(
    week_lookup["week_start"].dt.year.min(),
    week_lookup["week_end"].dt.year.max() + 1,
)

holiday_dates = {
    "thanksgiving_week": [
        fourth_thursday_of_november(year) for year in years
    ],
    "christmas_week": [
        pd.Timestamp(year=year, month=12, day=25) for year in years
    ],
    "new_year_week": [
        pd.Timestamp(year=year, month=1, day=1) for year in years
    ],
    "easter_week": [
        pd.Timestamp(easter(year)) for year in years
    ],
}

for variable, dates in holiday_dates.items():
    week_lookup[variable] = False
    for holiday_date in dates:
        contains_holiday = (
            week_lookup["week_start"].le(holiday_date)
            & week_lookup["week_end"].ge(holiday_date)
        )
        week_lookup.loc[contains_holiday, variable] = True


In [ ]:
assert len(week_lookup) == 399
assert week_lookup["week"].is_unique
assert week_lookup.loc[
    week_lookup["week"].eq(1), "week_start"
].iloc[0] == pd.Timestamp("1989-09-14")
assert week_lookup.loc[
    week_lookup["week"].eq(399), "week_end"
].iloc[0] == pd.Timestamp("1997-05-07")
assert (
    week_lookup["week_end"] - week_lookup["week_start"]
).eq(pd.Timedelta(days=6)).all()

main_data = main_data.merge(
    week_lookup,
    on="week",
    how="left",
    validate="many_to_one",
    indicator="calendar_merge",
)
assert main_data["calendar_merge"].eq("both").all()
main_data = main_data.drop(columns="calendar_merge")


## 6. Past-only pricing and promotion variables

The regular-price benchmark is the 90th percentile of available model prices in the exact preceding 13 calendar weeks. The current week is excluded. Promotion status combines the recorded `SALE` field with a discount of at least 5% relative to this past-only benchmark.


In [ ]:
main_data = (
    main_data
    .sort_values(["store", "upc", "week_end"])
    .reset_index(drop=True)
)

rolling_regular_price = (
    main_data[["store", "upc", "week_end", "model_unit_price"]]
    .groupby(["store", "upc"], observed=True, sort=False)
    .rolling(
        window=REGULAR_PRICE_WINDOW,
        on="week_end",
        closed="left",
        min_periods=REGULAR_PRICE_MIN_PERIODS,
    )["model_unit_price"]
    .quantile(REGULAR_PRICE_QUANTILE)
    .rename("regular_price")
    .reset_index()
)

# Verify alignment before assigning without a second full-data merge.
rolling_keys = rolling_regular_price[
    ["store", "upc", "week_end"]
].reset_index(drop=True)
main_keys = main_data[
    ["store", "upc", "week_end"]
].reset_index(drop=True)
assert rolling_keys.equals(main_keys)

main_data["regular_price"] = rolling_regular_price[
    "regular_price"
].to_numpy()

del rolling_regular_price, rolling_keys, main_keys


In [ ]:
main_data["discount_depth"] = (
    1 - main_data["model_unit_price"] / main_data["regular_price"]
).clip(lower=0)

main_data["promo_from_discount"] = main_data["discount_depth"].ge(
    DISCOUNT_THRESHOLD
)
main_data["promo_state"] = (
    main_data["promo_recorded"]
    | main_data["promo_from_discount"]
)

assert main_data["discount_depth"].dropna().ge(0).all()
assert (
    main_data.loc[main_data["promo_recorded"], "promo_state"]
).all()


In [ ]:
grouped = main_data.groupby(
    ["store", "upc"],
    observed=True,
    sort=False,
)

main_data["previous_week"] = grouped["week"].shift(1)
main_data["previous_promo"] = (
    grouped["promo_state"].shift(1).astype("boolean")
)

consecutive_previous_week = (
    main_data["week"] - main_data["previous_week"]
).eq(1)

main_data["post_promo"] = (
    consecutive_previous_week
    & main_data["previous_promo"].fillna(False)
    & ~main_data["promo_state"]
)

assert not (
    main_data["post_promo"] & main_data["promo_state"]
).any()


In [ ]:
main_data["pricing_state"] = pd.Categorical(
    np.select(
        [main_data["promo_state"], main_data["post_promo"]],
        ["promotion", "post_promotion"],
        default="regular",
    ),
    categories=["regular", "promotion", "post_promotion"],
)

main_data["log_price"] = np.log(
    main_data["model_unit_price"].where(
        main_data["model_unit_price"].gt(0)
    )
)
main_data["scaled_time_trend"] = main_data["time_trend"] / 100

# Integer identifier for store–UPC fixed effects. Treat it as categorical or
# absorb it as a fixed effect in the estimation code; do not use it as a slope.
main_data["store_upc"] = (
    main_data.groupby(["store", "upc"], observed=True, sort=False)
    .ngroup()
    .astype("int32")
)


## 7. Construct the eligible demand sample

The exact estimation sample is materialized only after all past-dependent variables have been constructed. UPC 317 is excluded because it is a Tony the Tiger T-shirt rather than cereal. Product labels are stored separately to avoid repeating strings over millions of rows.


In [ ]:
product_lookup = (
    main_data[["upc", "descrip", "size"]]
    .drop_duplicates(subset="upc")
    .sort_values("upc")
    .reset_index(drop=True)
)

excluded_products = product_lookup.loc[
    product_lookup["upc"].isin(NON_CEREAL_UPCS)
]
display(excluded_products)

assert set(excluded_products["upc"]) == NON_CEREAL_UPCS
assert excluded_products["descrip"].astype("string").str.contains(
    "T-SH", case=False, na=False
).all()

main_data["valid_cereal_product"] = ~main_data["upc"].isin(
    NON_CEREAL_UPCS
)


In [ ]:
demand_columns = [
    # Panel identifiers
    "store", "upc", "store_upc", "week", "week_start", "week_end",

    # Outcome and price
    "move", "model_unit_price", "unit_price_observed", "log_price",
    "regular_price", "discount_depth",

    # Price provenance
    "price_source", "price_imputed", "tier_week_price_donors",
    "price_tier",

    # Promotion state
    "sale", "promo_recorded", "promo_from_discount", "promo_state",
    "post_promo", "pricing_state",

    # Calendar controls
    "calendar_month", "scaled_time_trend", "thanksgiving_week",
    "christmas_week", "new_year_week", "easter_week",

    # Observed margin for later cost reconstruction
    "gross_margin_pct_observed",
]


In [ ]:
demand_sample_mask = (
    main_data["valid_cereal_product"]
    & main_data["move"].ge(0)
    & main_data["model_unit_price"].gt(0)
    & main_data["regular_price"].gt(0)
    & main_data["discount_depth"].notna()
    & np.isfinite(main_data["log_price"])
)

# Materialize the eligible estimation sample without an additional deep copy.
demand_data = main_data.loc[
    demand_sample_mask,
    demand_columns,
].copy(deep=False)

assert demand_data["price_source"].isin(
    ["observed", "tier_median_imputed"]
).all()
assert demand_data["price_imputed"].eq(
    demand_data["unit_price_observed"].isna()
).all()
assert demand_data.loc[
    demand_data["price_imputed"], "gross_margin_pct_observed"
].isna().all()

# main_data is no longer needed after the eligible sample is materialized.
del main_data


In [ ]:
eligible_sample_summary = pd.Series(
    {
        "rows": len(demand_data),
        "stores": demand_data["store"].nunique(),
        "products": demand_data["upc"].nunique(),
        "store_upc_pairs": demand_data["store_upc"].nunique(),
        "weeks": demand_data["week"].nunique(),
        "zero_sales_share": demand_data["move"].eq(0).mean(),
        "imputed_price_share": demand_data["price_imputed"].mean(),
        "recorded_promo_share": demand_data["promo_recorded"].mean(),
        "combined_promo_share": demand_data["promo_state"].mean(),
        "post_promo_share": demand_data["post_promo"].mean(),
    },
    name="value",
)

display(eligible_sample_summary)


## 8. Targeted sales-anomaly filter

The filter is deliberately narrow. A row is flagged only when:

- sales are at least 1,000 units;
- at least 20 stores report the same UPC–week;
- its sales exceed ten times the **leave-one-store-out** cross-store 95th percentile for that UPC–week.

The leave-one-out comparison prevents a candidate observation from inflating its own benchmark. The unfiltered eligible sample remains available as `demand_data` for sensitivity analysis.


In [ ]:
sales_outlier_candidates = demand_data.loc[
    demand_data["move"].ge(SALES_OUTLIER_MIN_MOVE),
    ["store", "upc", "week", "move"],
].copy()
sales_outlier_candidates["_row_index"] = sales_outlier_candidates.index

candidate_keys = sales_outlier_candidates[
    ["upc", "week"]
].drop_duplicates()

comparison_pool = demand_data[
    ["store", "upc", "week", "move"]
].merge(
    candidate_keys,
    on=["upc", "week"],
    how="inner",
    validate="many_to_many",
)


In [ ]:
candidate_comparisons = sales_outlier_candidates[
    ["_row_index", "store", "upc", "week", "move"]
].merge(
    comparison_pool,
    on=["upc", "week"],
    how="left",
    suffixes=("_candidate", "_comparison"),
    validate="many_to_many",
)

# Exclude the candidate store from its own cross-store reference distribution.
candidate_comparisons = candidate_comparisons.loc[
    candidate_comparisons["store_candidate"].ne(
        candidate_comparisons["store_comparison"]
    )
]

leave_one_out_stats = (
    candidate_comparisons.groupby("_row_index", observed=True)
    .agg(
        comparison_stores=("store_comparison", "nunique"),
        comparison_median_move=("move_comparison", "median"),
        comparison_p95_move=(
            "move_comparison",
            lambda x: x.quantile(0.95),
        ),
    )
    .reset_index()
)


In [ ]:
sales_outlier_candidates = sales_outlier_candidates.merge(
    leave_one_out_stats,
    on="_row_index",
    how="left",
    validate="one_to_one",
)
sales_outlier_candidates["reporting_stores"] = (
    sales_outlier_candidates["comparison_stores"] + 1
)

sales_outlier_candidates["move_to_p95"] = (
    sales_outlier_candidates["move"]
    / sales_outlier_candidates["comparison_p95_move"].clip(lower=1)
)
sales_outlier_candidates["move_to_median"] = (
    sales_outlier_candidates["move"]
    / sales_outlier_candidates["comparison_median_move"].clip(lower=1)
)

sales_outlier_candidates["sales_outlier"] = (
    sales_outlier_candidates["reporting_stores"].ge(
        SALES_OUTLIER_MIN_STORES
    )
    & sales_outlier_candidates["move_to_p95"].ge(
        SALES_OUTLIER_MIN_P95_RATIO
    )
)


In [ ]:
flagged_candidates = sales_outlier_candidates.loc[
    sales_outlier_candidates["sales_outlier"]
].copy()
flagged_row_indices = flagged_candidates["_row_index"].to_numpy()

demand_data["sales_outlier"] = False
demand_data.loc[flagged_row_indices, "sales_outlier"] = True

outlier_summary = pd.Series(
    {
        "flagged_rows": demand_data["sales_outlier"].sum(),
        "flagged_share": demand_data["sales_outlier"].mean(),
        "flagged_units": demand_data.loc[
            demand_data["sales_outlier"], "move"
        ].sum(),
        "flagged_stores": demand_data.loc[
            demand_data["sales_outlier"], "store"
        ].nunique(),
    },
    name="value",
)

display(outlier_summary)


In [ ]:
sales_outlier_audit = demand_data.loc[
    demand_data["sales_outlier"]
].copy()
sales_outlier_audit["_row_index"] = sales_outlier_audit.index

comparison_columns = [
    "_row_index", "reporting_stores", "comparison_median_move",
    "comparison_p95_move", "move_to_median", "move_to_p95",
]

sales_outlier_audit = (
    sales_outlier_audit
    .merge(product_lookup, on="upc", how="left", validate="many_to_one")
    .merge(
        flagged_candidates[comparison_columns],
        on="_row_index",
        how="left",
        validate="one_to_one",
    )
    .sort_values("move_to_p95", ascending=False)
)

display(sales_outlier_audit.head(50))

# Keep the clean model data lean; the all-sales sample remains in demand_data.
demand_model_data = demand_data.loc[
    ~demand_data["sales_outlier"],
    demand_columns,
].copy(deep=False)

assert len(demand_data) - len(demand_model_data) == int(
    outlier_summary["flagged_rows"]
)

del candidate_keys, comparison_pool, candidate_comparisons, leave_one_out_stats


## 9. Final quality assurance


In [ ]:
required_columns = [
    "store", "upc", "store_upc", "week", "move",
    "model_unit_price", "log_price", "regular_price",
    "discount_depth", "promo_recorded", "post_promo",
    "calendar_month", "scaled_time_trend",
    "thanksgiving_week", "christmas_week",
    "new_year_week", "easter_week",
]

missing_required = demand_model_data[required_columns].isna().sum()
display(missing_required.rename("missing_values"))

assert missing_required.eq(0).all()
assert not demand_model_data.duplicated(
    ["store", "upc", "week"]
).any()
assert demand_model_data["move"].ge(0).all()
assert demand_model_data["model_unit_price"].gt(0).all()
assert demand_model_data["regular_price"].gt(0).all()
assert demand_model_data["discount_depth"].ge(0).all()
assert demand_model_data["calendar_month"].between(1, 12).all()


In [ ]:
assert np.isfinite(demand_model_data["log_price"]).all()
assert np.allclose(
    demand_model_data["log_price"],
    np.log(demand_model_data["model_unit_price"]),
)
assert not demand_model_data["week"].isin(EXCLUDED_WEEKS).any()
assert not demand_model_data["upc"].isin(NON_CEREAL_UPCS).any()
assert not (
    demand_model_data["post_promo"]
    & demand_model_data["promo_state"]
).any()
assert demand_model_data["price_imputed"].eq(
    demand_model_data["unit_price_observed"].isna()
).all()
assert demand_model_data.loc[
    demand_model_data["price_imputed"], "gross_margin_pct_observed"
].isna().all()


In [ ]:
final_sample_summary = pd.Series(
    {
        "rows": len(demand_model_data),
        "stores": demand_model_data["store"].nunique(),
        "products": demand_model_data["upc"].nunique(),
        "store_upc_pairs": demand_model_data["store_upc"].nunique(),
        "weeks": demand_model_data["week"].nunique(),
        "zero_sales_share": demand_model_data["move"].eq(0).mean(),
        "imputed_price_share": demand_model_data["price_imputed"].mean(),
        "recorded_promo_share": demand_model_data["promo_recorded"].mean(),
        "post_promo_share": demand_model_data["post_promo"].mean(),
        "excluded_sales_outliers": outlier_summary["flagged_rows"],
    },
    name="value",
)

display(final_sample_summary)


In [ ]:
descriptive_columns = [
    "move", "model_unit_price", "regular_price", "discount_depth",
    "log_price", "scaled_time_trend", "gross_margin_pct_observed",
]

descriptive_summary = demand_model_data[
    descriptive_columns
].describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
).T

display(descriptive_summary)


## 10. Save processed data and audit tables

Large processed tables are saved as Parquet with Zstandard compression. Parquet is portable, supports selective column loading, and is preferable to pickle for reproducible research workflows. Small summaries remain CSV files so they can be inspected directly.


### 10.1 Save Parquet datasets


In [ ]:
# Preserve the eligible all-sales sample and the cleaned main sample.
DEMAND_DATA_PATH = PROCESSED_DIR / "cereal_demand_data.parquet"
DEMAND_MODEL_DATA_PATH = (
    PROCESSED_DIR / "cereal_demand_model_data.parquet"
)
PRODUCT_LOOKUP_PATH = PROCESSED_DIR / "cereal_product_lookup.parquet"
SALES_OUTLIER_PATH = PROCESSED_DIR / "cereal_sales_outliers.parquet"

demand_data.to_parquet(
    DEMAND_DATA_PATH,
    index=False,
    engine="pyarrow",
    compression=PARQUET_COMPRESSION,
)
demand_model_data.to_parquet(
    DEMAND_MODEL_DATA_PATH,
    index=False,
    engine="pyarrow",
    compression=PARQUET_COMPRESSION,
)
product_lookup.to_parquet(
    PRODUCT_LOOKUP_PATH,
    index=False,
    engine="pyarrow",
    compression=PARQUET_COMPRESSION,
)
sales_outlier_audit.to_parquet(
    SALES_OUTLIER_PATH,
    index=False,
    engine="pyarrow",
    compression=PARQUET_COMPRESSION,
)


### 10.2 Verify Parquet outputs


In [ ]:
parquet_outputs = {
    "eligible demand sample": (DEMAND_DATA_PATH, demand_data),
    "cleaned demand sample": (DEMAND_MODEL_DATA_PATH, demand_model_data),
    "product lookup": (PRODUCT_LOOKUP_PATH, product_lookup),
    "sales-outlier audit": (SALES_OUTLIER_PATH, sales_outlier_audit),
}

# Validate dimensions from file metadata without reloading the large tables.
for label, (path, frame) in parquet_outputs.items():
    parquet_file = pq.ParquetFile(path)
    assert parquet_file.metadata.num_rows == len(frame), label
    assert parquet_file.metadata.num_columns == len(frame.columns), label

# Read one row group to confirm that the main file is readable and ordered.
verification_columns = [
    "store", "upc", "week", "move", "model_unit_price",
]
verification_sample = (
    pq.ParquetFile(DEMAND_MODEL_DATA_PATH)
    .read_row_group(0, columns=verification_columns)
    .to_pandas()
    .head(100)
    .reset_index(drop=True)
)

pd.testing.assert_frame_equal(
    verification_sample,
    demand_model_data[verification_columns].head(100).reset_index(drop=True),
    check_dtype=False,
)

print(f"Parquet round-trip verified with PyArrow {pa.__version__}.")


### 10.3 Save human-readable summaries


In [ ]:
final_sample_summary.to_csv(
    TABLE_DIR / "cereal_demand_sample_summary.csv",
    header=True,
)
validation_summary.to_csv(
    TABLE_DIR / "price_imputation_validation_summary.csv",
    header=True,
)
validation_by_donors.to_csv(
    TABLE_DIR / "price_imputation_validation_by_donors.csv"
)
validation_by_tier.to_csv(
    TABLE_DIR / "price_imputation_validation_by_tier.csv"
)
outlier_summary.to_csv(
    TABLE_DIR / "sales_outlier_summary.csv",
    header=True,
)
descriptive_summary.to_csv(
    TABLE_DIR / "cereal_demand_descriptive_summary.csv"
)

print("Saved final demand sample:", DEMAND_MODEL_DATA_PATH)
print("Final shape:", demand_model_data.shape)


## Reference

```bibtex
@techreport{mehrhoff2018dominicks,
  author      = {Mehrhoff, Jens},
  title       = {Promoting the Use of a Publicly Available Scanner Data Set
                 in Price Index Research and for Capacity Building},
  institution = {European Commission},
  year        = {2018}
}
```
